In [1]:
import pymc as pm
import pytensor
import pytensor.tensor as pt
import jax
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from numpy.lib.stride_tricks import sliding_window_view
import matplotlib.pyplot as plt
import os
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from scipy.spatial.distance import pdist
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Dense, Conv1D, BatchNormalization, Bidirectional, LSTM, Concatenate, Activation, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import Huber, MeanSquaredError, LogCosh, MeanAbsoluteError
from tensorflow.keras.initializers import GlorotUniform, Orthogonal
import random
import json
from scipy.stats import zscore

def extract_hrv_features_2(serie, window_size=20, window_size_long=40):
    """
    Toma una serie de intervalos RR y los tamaños de ventana para construir
    un DataFrame de características ortogonales y la variable objetivo (target).
    """
    if window_size_long < window_size:
        raise ValueError("window_size_long debe ser mayor o igual a window_size")

    serie = np.asarray(serie, dtype=float)

    # 1. Generar ventanas sobre la serie COMPLETA
    ventanas_long = sliding_window_view(serie, window_size_long)

    # 2. Separar variables predictoras del target
    X_ventanas_long = ventanas_long[:-1]
    y_target = serie[window_size_long:]

    # 3. Extraer ventana CORTA
    X_ventanas_short = X_ventanas_long[:, -window_size:]

    # 4. Calcular diferencias dentro de la ventana CORTA
    diffs = np.diff(X_ventanas_short, axis=1)

    # ---- MÉTRICAS TRADICIONALES ----
    n_above = np.sum(diffs > 0, axis=1)
    n_below = np.sum(diffs < 0, axis=1)
    suma_porta = n_above + n_below
    porta_index = np.divide(n_below, suma_porta, out=np.zeros_like(n_below, dtype=float), where=suma_porta!=0)

    d_above = np.sum(np.abs(diffs) * (diffs > 0), axis=1) / np.sqrt(2)
    d_total = np.sum(np.abs(diffs), axis=1) / np.sqrt(2)
    guzic_index = np.divide(d_above, d_total, out=np.zeros_like(d_above, dtype=float), where=d_total!=0)

    nn50 = np.sum(np.abs(diffs) > 50, axis=1)
    nn20 = np.sum(np.abs(diffs) > 20, axis=1)

    sdsd = np.std(diffs, axis=1)
    sd1 = np.sqrt((sdsd**2) / 2)

    mean_val = np.mean(X_ventanas_short, axis=1)
    std_val = np.std(X_ventanas_short, axis=1)
    var_val = std_val ** 2

    std_long = np.std(X_ventanas_long, axis=1)
    inner_value = 2 * std_long**2 - sd1**2
    sd2 = np.sqrt(np.maximum(inner_value, 0))
    c_n = np.pi * sd1 * sd2

    # CCM
    ventanas_4puntos = sliding_window_view(X_ventanas_short, window_shape=4, axis=1)
    rr_i, rr_i1, rr_i2, rr_i3 = ventanas_4puntos[:, 0], ventanas_4puntos[:, 1], ventanas_4puntos[:, 2], ventanas_4puntos[:, 3]
    areas = 0.5 * np.abs(rr_i * (rr_i2 - rr_i3) - rr_i1 * (rr_i1 - rr_i3) + rr_i2 * (rr_i1 - rr_i2))
    denominador_ccm = c_n * (window_size - 2)
    ccm = np.divide(np.sum(areas, axis=1), denominador_ccm, out=np.zeros_like(c_n), where=denominador_ccm!=0)
    ccm = np.where(ccm > 1, 1, ccm)

    # -------------------------------------------------------------
    # NUEVAS CARACTERÍSTICAS ORTOGONALES (NO COLINEALES)
    # -------------------------------------------------------------

    # A. Coeficiente de Variación (CV)
    cv = np.divide(std_val, mean_val, out=np.zeros_like(std_val, dtype=float), where=mean_val!=0)

    # B. Robustez a Outliers: Rango Intercuartílico (IQR) y MAD
    q75, q25 = np.percentile(X_ventanas_short, [75, 25], axis=1)
    iqr = q75 - q25

    median_val = np.median(X_ventanas_short, axis=1)
    mad = np.median(np.abs(X_ventanas_short - median_val[:, None]), axis=1)

    # C. Fragmentación del Ritmo Cardíaco (PIP - Puntos de Inflexión)
    diffs_1 = diffs[:, :-1]
    diffs_2 = diffs[:, 1:]
    inflections = (diffs_1 * diffs_2) <= 0
    pip = np.sum(inflections, axis=1) / (window_size - 2)

    # D. Asimetría (Skewness) de las diferencias
    mean_diffs = np.mean(diffs, axis=1, keepdims=True)
    std_diffs = np.std(diffs, axis=1, keepdims=True)
    std_diffs_safe = np.where(std_diffs == 0, 1e-10, std_diffs) # Evitar NaN
    skewness = np.mean(((diffs - mean_diffs) / std_diffs_safe)**3, axis=1)

    # -------------------------------------------------------------
    # EMPAQUETADO FINAL
    # -------------------------------------------------------------
    rr_columns = {f'rr_{i+1}': X_ventanas_short[:, i] for i in range(window_size)}

    stats_columns = {
        'n_above': n_above, 'n_below': n_below, 'nn20': nn20, 'nn50': nn50,
        'sdsd': sdsd, 'mean': mean_val, 'std': std_val, 'var': var_val,
        'std_long': std_long, 'sd1': sd1, 'sd2': sd2, 'c_n': c_n,
        'ccm': ccm, 'porta': porta_index, 'guzik': guzic_index,
        'cv': cv, 'iqr': iqr, 'mad': mad, 'pip': pip, 'skewness': skewness, # <--- NUEVAS
        'target': y_target
    }

    df = pd.DataFrame({**rr_columns, **stats_columns})
    return df

I0000 00:00:1782332631.884785 3959687 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1782332633.836705 3959687 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
# Cargar los datos
series_list = 'series/'
files = ['006.txt',
        '16265.txt',
        '16273.txt',
        '16786.txt',
        '16795.txt',
        '17453.txt',
        '18177.txt',
        '18184.txt',
        '16539.txt',
        'nsr054RRcl.txt',
        '005.txt',
        '16420.txt',
        '19140.txt',
        'nsr047RRcl.txt',
        'nsr048RRcl.txt',
        'nsr049RRcl.txt',
        '008.txt',
        '009.txt',
        '013.txt']
window_size = 20
np.random.seed(42)
SEED = 53
# select 50 random files
files_train = np.random.choice(files, size=11, replace=False) # np.array(['16795.txt', '013.txt', '18177.txt', 'nsr047RRcl.txt', '006.txt', '16786.txt', '19140.txt', 'nsr054RRcl.txt', '16420.txt', 'nsr048RRcl.txt'])

# the rest of the files will be used for testing
files_test = [f for f in files if f not in files_train]

feature_cols = ['mean', 'sdsd', 'sd2', 'ccm', 'guzik', 'nn50', 'porta', 'std']
rr_cols = [f'rr_{i}' for i in range(1, 21)]
rr_last_col = 'rr_20'

# --- 2. ESCALADO Y RESHAPE ---
scaler_feats = StandardScaler()

In [3]:
# 1. Define paths
save_dir = ''
model_path = os.path.join(save_dir, 'cnn_lstm_hybrid.keras')
json_path = os.path.join(save_dir, 'scalers_params.json')

# 2. Load the Keras Model
print("Loading model...")
loaded_model = load_model(model_path)
print("✅ Model loaded successfully.")

# 3. Load the Scaler Parameters
print("Loading scalers...")
with open(json_path, 'r') as json_file:
    loaded_scalers = json.load(json_file)

# 4. Extract parameters back into Numpy arrays (using float32 for Keras compatibility)
# Sequence Scaler (Global Mean/Scale)
seq_mean = np.array(loaded_scalers['scaler_seq']['mean'], dtype=np.float32)
seq_scale = np.array(loaded_scalers['scaler_seq']['scale'], dtype=np.float32)

# Features Scaler (8 features)
feats_mean = np.array(loaded_scalers['scaler_feats']['mean'], dtype=np.float32)
feats_scale = np.array(loaded_scalers['scaler_feats']['scale'], dtype=np.float32)

# Target Scaler (Delta RR)
y_mean = np.float32(loaded_scalers['y_scaler']['mean'][0])
y_scale = np.float32(loaded_scalers['y_scaler']['scale'][0])
print("✅ Scaler parameters reconstructed.")

Loading model...


E0000 00:00:1782332635.668009 3959687 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1782332635.668379 3959759 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
E0000 00:00:1782332635.680927 3959687 cuda_executor.cc:1827] Nvml call failed with 3(Not Supported). Assuming PCIe gen 3 x16 bandwidth.
W0000 00:00:1782332635.681645 3959687 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


✅ Model loaded successfully.
Loading scalers...
✅ Scaler parameters reconstructed.


In [ ]:
# --- 1. PREPARACIÓN DE DATOS DE PRUEBA ---
df_test = pd.DataFrame()
for file in files_test:
    data = np.loadtxt(os.path.join(series_list, file), dtype=int)
    df_temp = extract_hrv_features_2(data, window_size, window_size_long=40)
    df_temp['file_id'] = file
    df_test = pd.concat([df_test, df_temp], ignore_index=True)

# [Extracción asumiendo df_test ya preparado]
X_feats_test = df_test[feature_cols].values
X_rr_seq_test = df_test[rr_cols].values  # Matriz de Nx20
y_test_raw = df_test['target'].values
rr_last_test = df_test[rr_last_col].values

print(f"Preparing {X_rr_seq_test.shape[0]} rows for inference...")

# ---------------------------------------------------------
# 1. MANUAL SCALING (x_scaled = (x - mean) / scale)
# ---------------------------------------------------------
# Scale the static features
X_feats_test_scaled = (X_feats_test - feats_mean) / feats_scale

# Scale the sequences globally
X_rr_seq_test_scaled = (X_rr_seq_test - seq_mean) / seq_scale

# Reshape the sequence for the CNN-LSTM (N, 20, 1)
X_rr_seq_test_3d = X_rr_seq_test_scaled.reshape(-1, 20, 1)

# ---------------------------------------------------------
# 2. PREDICTION
# ---------------------------------------------------------
# Pass the list of inputs exactly as you did during training
y_pred_diff_scaled = loaded_model.predict([X_rr_seq_test_3d, X_feats_test_scaled], batch_size=50000)

# Flatten the output from (N, 1) to (N,)
y_pred_diff_scaled = y_pred_diff_scaled.flatten()

# ---------------------------------------------------------
# 3. MANUAL INVERSE SCALING & RECONSTRUCTION
# ---------------------------------------------------------
# Descale the prediction back to milliseconds (y_original = (y_scaled * scale) + mean)
y_pred_diff_ms = (y_pred_diff_scaled * y_scale) + y_mean

# Reconstruct the absolute next RR interval
y_pred_ms = y_pred_diff_ms + rr_last_test

print("✅ Predictions finished and descaled successfully!")

In [ ]:
# --- 5. EVALUACIÓN DE MÉTRICAS GLOBALES ---
mae = mean_absolute_error(y_test_raw, y_pred_ms)
rmse = np.sqrt(mean_squared_error(y_test_raw, y_pred_ms))
r2 = r2_score(y_test_raw, y_pred_ms)

print(f"\n--- Métricas en Test de No Vistos (CNN-LSTM Hybrid) ---")
print(f"MAE:  {mae:.2f} ms")
print(f"RMSE: {rmse:.2f} ms")
print(f"R²:   {r2:.4f}")

# --- 6. EVALUACIÓN DEL OBJETIVO DE NEGOCIO ---
errores_absolutos = np.abs(y_test_raw - y_pred_ms)
latidos_en_tolerancia = np.mean(errores_absolutos <= 15.0)

In [4]:
# here we're going to study how it change the correlation between the predicted and the real values when we eliminate some values from the original series

from utils import cxk

original_serie = np.loadtxt(os.path.join(series_list, files_test[0]), dtype=int).astype(float)

# percent to eliminate from the original series to create a test set
percent_to_eliminate = 0.1  # 20% of the original series
# the removing indexes will be selected randomly begining from 40 because the first 40 values are used to create the features and the target for the model
# 3. Define the start and end indices
n_to_eliminate = int(len(original_serie) * percent_to_eliminate)
start_idx = 40
random_indexes = random.sample(range(start_idx, len(original_serie)), n_to_eliminate)
# now we can substitute the values from the original series for Nan
modified_serie = original_serie.copy()
modified_serie[random_indexes] = np.nan

In [5]:
print("Starting autoregressive imputation...")

# Iterate over the series starting from index 40
for i in range(40, len(modified_serie)):
    
    # Check if the current value is a NaN
    if np.isnan(modified_serie[i]):
        
        # 1. EXTRAER ESTRICTAMENTE EL HISTORIAL (Exactamente 40 elementos)
        # modified_serie[i] NO está incluido aquí.
        history_clean = modified_serie[i-40 : i]
        
        # 2. ADAPTACIÓN PARA LA FUNCIÓN
        # Le pegamos un NaN al final para que extract_hrv_features_2 lo tome 
        # como 'y_target' y nos deje procesar el 'history_clean' sin errores.
        window_data_for_func = np.append(history_clean, np.nan)
        
        # 3. FEATURE EXTRACTION
        df_step = extract_hrv_features_2(window_data_for_func, window_size=20, window_size_long=40)
        
        X_feats_step = df_step[feature_cols].values
        X_rr_seq_step = df_step[rr_cols].values
        
        # 4. MANUAL SCALING
        X_feats_step_scaled = (X_feats_step - feats_mean) / feats_scale
        X_rr_seq_step_scaled = (X_rr_seq_step - seq_mean) / seq_scale
        
        # Reshape to 3D for the CNN-LSTM -> Shape: (1, 20, 1)
        X_rr_seq_step_3d = X_rr_seq_step_scaled.reshape(1, 20, 1)
        
        # 5. PREDICTION
        # verbose=0 prevents the loading bar from printing on every loop iteration
        y_pred_scaled = loaded_model.predict([X_rr_seq_step_3d, X_feats_step_scaled], verbose=0)
        
        # 6. DESCALING (Absolute Target)
        y_pred_ms = (y_pred_scaled.flatten()[0] * y_scale) + y_mean
        
        # 7. IMPUTATION
        # Inyectamos el valor predicho en la serie. 
        # En la siguiente iteración, este valor formará parte de 'history_clean'
        modified_serie[i] = y_pred_ms

print("✅ Imputation complete! All NaNs have been substituted.")

Starting autoregressive imputation...


KeyboardInterrupt: 